# YOLOv8s 객체 탐지 학습

- 설명: YOLOv8s 데이터 준비, 학습 및 평가 실험입니다.
- 작성자: 김동혁

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


# YOLOv8 학습 - Roboflow Dataset

In [ ]:
# =========================
# 1. 환경 확인
# =========================

import os
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU를 사용할 수 없습니다. Colab 런타임 유형을 GPU로 변경하세요.")

In [ ]:
# =========================
# 2. 패키지 설치
# =========================

!pip install -q roboflow ultralytics

In [ ]:
# =========================
# 3. Roboflow 데이터셋 다운로드
# =========================

import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY 환경변수가 설정되지 않았습니다.")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("korean-bus").project("test-68u3i")
version = project.version(2)

dataset = version.download("yolov8")

print("Dataset downloaded to:", dataset.location)

In [ ]:
# =========================
# 4. 데이터셋 구조 확인
# =========================

import os

dataset_path = dataset.location

print("Dataset path:", dataset_path)
print(os.listdir(dataset_path))

data_yaml_path = os.path.join(dataset_path, "data.yaml")
print("data.yaml path:", data_yaml_path)

In [ ]:
# =========================
# 5. data.yaml 내용 확인
# =========================

with open(data_yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# =========================
# 6. YOLOv8 모델 학습
# =========================

from ultralytics import YOLO

# 기본 모델 선택
# yolov8n.pt: 가장 가벼움, 빠름
# yolov8s.pt: 조금 더 정확하지만 느림
model = YOLO("yolov8m.pt")

results = model.train(
    data=data_yaml_path,
    epochs=100,           # 학습 횟수 증가 (50 -> 100)
    imgsz=640,            # 기본 이미지 크기 유지
    batch=8,             # T4 GPU에서 가장 안정적인 배치 크기
    name="hook_yolov8m", # 저장 폴더 이름 변경 (이전 결과와 섞임 방지)
    patience=20,          # 조기 종료 대기 횟수 증가 (10 -> 20)
    optimizer="auto",
    workers=2,
    lr0=0.01,
    cos_lr=True,
    device=0
)

In [ ]:
# =========================
# 7. 학습 결과 확인
# =========================

from IPython.display import Image, display
import glob

result_dir = "runs/detect/hook_yolov8m"

# 학습 곡선
results_png = os.path.join(result_dir, "results.png")
if os.path.exists(results_png):
    display(Image(filename=results_png))

# Confusion matrix
confusion_matrix = os.path.join(result_dir, "confusion_matrix.png")
if os.path.exists(confusion_matrix):
    display(Image(filename=confusion_matrix))

In [ ]:
# =========================
# 8. 검증 데이터로 평가
# =========================

best_model_path = os.path.join(result_dir, "weights", "best.pt")

trained_model = YOLO(best_model_path)

metrics = trained_model.val(
    data=data_yaml_path,
    imgsz=640,
    device=0
)

print(metrics)

In [ ]:
# =========================
# 9. 학습된 모델 다운로드
# =========================

from google.colab import files

# 파일이 실제로 존재하는지 확인 후 다운로드 진행
if os.path.exists(best_model_path):
    print(f"🎉 최적의 모델 파일을 찾았습니다: {best_model_path}")
    print("로컬 PC로 다운로드를 시작합니다. 잠시만 기다려주세요...")
    files.download(best_model_path)
else:
    print("❌ 지정된 경로에 best.pt 파일이 없습니다. 학습이 아직 안 끝났거나 폴더명을 확인해 주세요.")

In [ ]:
# =========================
# 10. 테스트 이미지 예측
# =========================

test_images_dir = os.path.join(dataset_path, "test", "images")

pred_results = trained_model.predict(
    source=test_images_dir,
    imgsz=640,
    conf=0.5,
    save=True
)

print("Prediction completed.")

In [ ]:
# =========================
# 11. 예측 결과 이미지 확인
# =========================

import glob
from IPython.display import Image, display

predict_dirs = sorted(glob.glob("runs/detect/predict*"))
latest_predict_dir = predict_dirs[-1]

predicted_images = glob.glob(os.path.join(latest_predict_dir, "*.jpg")) + \
                   glob.glob(os.path.join(latest_predict_dir, "*.png"))

print("Prediction result dir:", latest_predict_dir)
print("Number of predicted images:", len(predicted_images))

for img_path in predicted_images[:10]:
    display(Image(filename=img_path))

# 새로운 이미지로 YOLOv8 탐지 테스트

In [ ]:
# =========================
# 1. 테스트할 이미지 업로드
# =========================

from google.colab import files

uploaded = files.upload()

uploaded_image_paths = list(uploaded.keys())
print("Uploaded images:", uploaded_image_paths)

In [ ]:
# =========================
# 2. 학습된 모델 불러오기
# =========================

from ultralytics import YOLO
import os

best_model_path = "runs/detect/hook_yolov8m/weights/best.pt"

model = YOLO(best_model_path)

In [ ]:
# =========================
# 3. 새 이미지 탐지 실행
# =========================

results = model.predict(
    source=uploaded_image_paths,
    imgsz=640,
    conf=0.1,
    save=True
)

print("Detection completed.")

In [ ]:
# =========================
# 4. 탐지 결과 이미지 확인
# =========================

import glob
from IPython.display import Image, display

predict_dirs = sorted(glob.glob("runs/detect/predict*"))
latest_predict_dir = predict_dirs[-1]

result_images = glob.glob(os.path.join(latest_predict_dir, "*.jpg")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.png")) + \
                glob.glob(os.path.join(latest_predict_dir, "*.jpeg"))

print("Result dir:", latest_predict_dir)

for img_path in result_images:
    display(Image(filename=img_path))